# Literature Database Builder With Source Registry

This notebook is a source-transparent literature database builder.

It is designed to make the workflow auditable, so you can clearly show:

1. which papers and datasets were included
2. which URLs or PDFs were used as inputs
3. which structured literature records were extracted
4. which formatted references should be cited in presentations or reports


## Workflow Visibility Goal

This notebook keeps a clean separation between:

- source registration
- database building
- paper visibility and provenance review
- formatted reference export

The aim is that the top of the workflow should always make it easy to answer:

- What literature is currently included?
- What is each paper used for?
- What title / DOI / paper ID corresponds to each extracted evidence record?


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display, Markdown

project_dir = Path.cwd()
sys.path.append(str(project_dir))

from neurodesk_literature_to_pgsql import analyze_sources, build_evidence_framework


## Stage 1. Register Literature Inputs

Edit this list whenever you add or remove sources.

Recommended fields:

- `source_label`: short human-readable label
- `input_value`: URL or local PDF path
- `source_group`: source family or topic
- `disease_focus`: main disease or syndrome supported
- `note`: why the source is included


In [ ]:
LITERATURE_SOURCES = [
    {
        'source_label': 'PPA variants paper',
        'input_value': 'https://pubmed.ncbi.nlm.nih.gov/21810890/',
        'source_group': 'FTD-spectrum language variants',
        'disease_focus': 'PPA / svPPA / nfvPPA / lvPPA',
        'note': 'Subtype-specific language-led FTD evidence.'
    },
    {
        'source_label': 'bvFTD diagnostic criteria',
        'input_value': 'https://pubmed.ncbi.nlm.nih.gov/21325651/',
        'source_group': 'FTD-spectrum behavioural variant',
        'disease_focus': 'bvFTD',
        'note': 'Behavioural-variant FTD evidence and diagnostic framing.'
    },
    {
        'source_label': 'OASIS-3 dataset page',
        'input_value': 'https://sites.wustl.edu/oasisbrains/home/oasis-3/',
        'source_group': 'Dataset reference',
        'disease_focus': 'Aging / dementia dataset context',
        'note': 'Describes OASIS-3 dataset provenance and acquisition context.'
    },
    {
        'source_label': 'OASIS overview paper',
        'input_value': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC3776065/',
        'source_group': 'Dataset reference',
        'disease_focus': 'OASIS cohort context',
        'note': 'Supports explanation of the OASIS imaging dataset and cohort structure.'
    },
    {
        'source_label': 'OASIS-3 related paper',
        'input_value': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC4058576/',
        'source_group': 'Dataset reference',
        'disease_focus': 'Longitudinal dementia imaging context',
        'note': 'Supports data-source narrative for longitudinal neuroimaging.'
    },
    {
        'source_label': 'ADNI data page',
        'input_value': 'https://adni.loni.usc.edu/data-samples/adni-data/',
        'source_group': 'Dataset reference',
        'disease_focus': 'AD / MCI dataset context',
        'note': 'Supports ADNI dataset provenance and comparison context.'
    },
    {
        'source_label': 'ADNI methods paper A',
        'input_value': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC3809512/',
        'source_group': 'Dataset reference',
        'disease_focus': 'ADNI cohort / imaging context',
        'note': 'Supports structured description of ADNI-related imaging context.'
    },
    {
        'source_label': 'ADNI methods paper B',
        'input_value': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC2733354/',
        'source_group': 'Dataset reference',
        'disease_focus': 'ADNI biomarker / imaging framework',
        'note': 'Background reference for ADNI design and usage.'
    },
    {
        'source_label': 'ADNI methods paper C',
        'input_value': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC2714061/',
        'source_group': 'Dataset reference',
        'disease_focus': 'ADNI imaging workflow context',
        'note': 'Supports study-design and acquisition-context transparency.'
    },
    {
        'source_label': 'ADNI methods paper D',
        'input_value': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC2827264/',
        'source_group': 'Dataset reference',
        'disease_focus': 'ADNI MRI context',
        'note': 'Additional ADNI MRI reference for workflow provenance.'
    },
    {
        'source_label': 'ADNI methods paper E',
        'input_value': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC4754499/',
        'source_group': 'Dataset reference',
        'disease_focus': 'ADNI longitudinal context',
        'note': 'Additional ADNI longitudinal imaging reference.'
    },
]

OUTPUT_DIR = Path('./literature_registry_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_REGISTRY_CSV_PATH = OUTPUT_DIR / 'literature_source_registry.csv'
SOURCE_REGISTRY_JSON_PATH = OUTPUT_DIR / 'literature_source_registry.json'
FORMATTED_REFERENCES_MD_PATH = OUTPUT_DIR / 'formatted_reference_list.md'

AI_BACKEND = 'none'
AI_BASE_URL = 'http://127.0.0.1:11434/v1'
AI_MODEL = 'llama3.1:8b'
AI_API_KEY = None


In [ ]:
def build_source_registry_df(sources):
    rows = []
    for idx, item in enumerate(sources, start=1):
        input_value = str(item.get('input_value', '')).strip()
        if input_value.startswith('http://') or input_value.startswith('https://'):
            source_type = 'web_url'
        elif input_value.lower().endswith('.pdf'):
            source_type = 'local_pdf'
        else:
            source_type = 'other'
        rows.append({
            'registry_id': idx,
            'source_label': item.get('source_label'),
            'input_value': input_value,
            'source_type': source_type,
            'source_group': item.get('source_group'),
            'disease_focus': item.get('disease_focus'),
            'note': item.get('note'),
        })
    return pd.DataFrame(rows)


source_registry_df = build_source_registry_df(LITERATURE_SOURCES)
source_registry_df.to_csv(SOURCE_REGISTRY_CSV_PATH, index=False)
SOURCE_REGISTRY_JSON_PATH.write_text(
    json.dumps(source_registry_df.to_dict(orient='records'), indent=2, ensure_ascii=False),
    encoding='utf-8',
)

display(Markdown('## Current Registered Source Inputs'))
display(source_registry_df)


## Stage 2. Top-Level Paper Visibility

This section should stay near the top of the workflow so that the current included papers are always visible.


In [ ]:
display(Markdown('## Current Included Papers (Input Registry View)'))
display(source_registry_df[['registry_id', 'source_label', 'input_value', 'source_group', 'disease_focus', 'note']])


## Stage 3. Build Database From Registered Sources

This step uses the source registry as the single source of truth.

Generated outputs:

- `literature_database.sql`
- `literature_database.sqlite`
- `literature_database.json`
- `literature_database.xlsx`


In [ ]:
INPUTS = source_registry_df['input_value'].dropna().astype(str).tolist()

build_result = analyze_sources(
    inputs=INPUTS,
    output_dir=OUTPUT_DIR,
    ai_backend=AI_BACKEND,
    ai_base_url=AI_BASE_URL,
    ai_model=AI_MODEL,
    ai_api_key=AI_API_KEY,
)

records_df = pd.DataFrame(build_result['records'])
if len(records_df) > 0:
    records_df = records_df.copy()
    records_df['paper_id'] = range(1, len(records_df) + 1)
    records_df['source_title'] = records_df['title']

display(Markdown('### Database build result'))
display(pd.DataFrame([
    {'item': 'n_registered_sources', 'value': len(INPUTS)},
    {'item': 'n_extracted_records', 'value': len(build_result['records'])},
    {'item': 'sql_path', 'value': build_result.get('sql_path')},
    {'item': 'sqlite_path', 'value': build_result.get('sqlite_path')},
    {'item': 'json_path', 'value': build_result.get('json_path')},
    {'item': 'xlsx_path', 'value': build_result.get('xlsx_path')},
]))


## Stage 4. Current Included Papers (Extracted Record View)

This is the table you can show at the top of the workflow when you want title / DOI / paper ID visibility.


In [ ]:
display(Markdown('## Current Included Papers (Extracted Record View)'))
paper_preview_cols = [
    c for c in [
        'paper_id',
        'source_label',
        'source_title',
        'year',
        'doi',
        'study_design',
        'diagnoses',
        'rois',
        'imaging_metrics',
        'source_path_or_url',
    ]
    if c in records_df.columns
]
display(records_df[paper_preview_cols])


## Stage 5. Evidence Linkage Table

This linkage table is the main provenance view.

Every evidence-linked row should clearly show:

- `paper_id`
- `source_title`
- `doi`


In [ ]:
evidence_df = build_evidence_framework(build_result['records'])

if len(records_df) > 0:
    merged_df = source_registry_df.merge(
        records_df,
        left_on='input_value',
        right_on='source_path_or_url',
        how='left'
    )
else:
    merged_df = source_registry_df.copy()

display(Markdown('### Source-to-record linkage table'))
display(merged_df[[
    c for c in [
        'registry_id',
        'paper_id',
        'source_label',
        'source_title',
        'doi',
        'input_value',
        'disease_focus',
        'diagnoses',
        'rois',
        'imaging_metrics',
    ]
    if c in merged_df.columns
]])

display(Markdown('### Evidence framework preview with source_title / doi / paper_id'))
display(evidence_df[[
    c for c in [
        'paper_id',
        'source_title',
        'doi',
        'diagnosis',
        'roi_name',
        'imaging_metric',
        'pattern_summary',
        'limitations',
    ]
    if c in evidence_df.columns
]].head(50))


## Stage 6. Formatted Reference List

This produces a readable reference list for supervision, slide decks, and appendices.


In [ ]:
def format_reference_row(row):
    authors = row.get('authors') or []
    if isinstance(authors, str):
        authors_text = authors
    else:
        authors_text = ', '.join([str(a) for a in authors if str(a).strip()])
    year = row.get('year') or 'n.d.'
    title = row.get('source_title') or row.get('title') or 'Untitled source'
    doi = row.get('doi')
    url = row.get('source_path_or_url')
    paper_id = row.get('paper_id')

    parts = [f'[{paper_id}]']
    if authors_text:
        parts.append(f'{authors_text} ({year}).')
    else:
        parts.append(f'({year}).')
    parts.append(f'{title}.')
    if doi:
        parts.append(f'DOI: {doi}.')
    if url:
        parts.append(f'URL: {url}')
    return ' '.join(parts)


formatted_references = []
if len(records_df) > 0:
    formatted_references = [format_reference_row(row) for _, row in records_df.iterrows()]

FORMATTED_REFERENCES_MD_PATH.write_text(
    '# Formatted Reference List\n\n' + '\n'.join([f'- {ref}' for ref in formatted_references]),
    encoding='utf-8',
)

display(Markdown('## Formatted Reference List'))
display(Markdown('\n'.join([f'- {ref}' for ref in formatted_references]) if formatted_references else '- No references extracted yet.'))


## Stage 7. Presentation-Friendly Summary

This final summary is designed for quick reporting to a supervisor.


In [ ]:
summary_lines = [
    '# Literature Workflow Source Summary',
    '',
    f'- Number of registered source inputs: {len(source_registry_df)}',
    f"- Number of extracted paper records: {len(records_df)}",
    f"- Source registry CSV: `{SOURCE_REGISTRY_CSV_PATH}`",
    f"- Source registry JSON: `{SOURCE_REGISTRY_JSON_PATH}`",
    f"- Formatted references markdown: `{FORMATTED_REFERENCES_MD_PATH}`",
    f"- Structured database SQL: `{build_result.get('sql_path')}`",
    f"- Structured database SQLite: `{build_result.get('sqlite_path')}`",
    f"- Structured database JSON: `{build_result.get('json_path')}`",
    f"- Structured database XLSX: `{build_result.get('xlsx_path')}`",
    '',
    '## Current included papers',
]

if len(records_df) > 0:
    for _, row in records_df.iterrows():
        summary_lines.append(
            f"- paper_id={row.get('paper_id')} | title={row.get('source_title')} | doi={row.get('doi')} | source={row.get('source_path_or_url')}"
        )
else:
    summary_lines.append('- No extracted records yet.')

display(Markdown('\n'.join(summary_lines)))
